# Challenge 2: Can We Trust the Spam Filter?

Your team is building a prototype email filter. **Spam is class 1; legitimate email is class 0.** Flagging a legitimate message could hide an important email. Missing some spam is inconvenient, but blocking good mail is the bigger concern.

**Your mission:** build a Gaussian Naive Bayes baseline, compare regularized logistic regression models with **3-, 4-, and 5-fold cross-validation**, and recommend a model using evidence—not accuracy alone.

## Start in two minutes
1. In Colab, choose **File → Save a copy in Drive**. Add your team name to the filename.
2. Work from top to bottom. Run supplied cells with **Shift+Enter**. Complete the four small **YOUR TURN** cells and make the decisions together.
3. Write your **six numbered answers in one Word document**, not in a separate PDF. Keep code, tables and figures in this notebook.
4. One member submits **`Challenge-2_TeamName.zip`** to Moodle, containing the **`.docx` answers and completed `.ipynb` notebook**.

**75-minute plan:** load and split (8 min) → cross-validated baseline (12 min) → regularization and overfitting plots (25 min) → choose and test (10 min) → write and submit (20 min). Start writing each answer when you reach it. The comparison loop and plotting code are supplied so you can focus on interpretation.

**Team name:** …  
**Members:** …

You may use the course Thinking Partner for syntax and debugging. Make a prediction first, check the suggested code and explain the results yourselves. No AI transcript is required. If stuck for more than three minutes, ask for help and move on with the supplied support code.

## 1. Understand the decision and load the data · 0–8 minutes

**Before running:** which error is more harmful here: a false positive or a false negative?

- **Precision:** of messages flagged as spam, what fraction really are spam? `TP / (TP + FP)`.
- **Recall:** of all actual spam messages, what fraction did we catch? `TP / (TP + FN)`.
- **Accuracy:** what fraction of all messages did we classify correctly?

Choose and justify the main metric for this scenario. To avoid a filter that flags almost nothing, our classroom requirement is **mean cross-validation recall ≥ 0.80**. Among models meeting that requirement, prioritize **mean cross-validation precision**. This is a teaching requirement, not an established real-world policy.

**Q1 — Problem and metric.** Explain a false positive and a false negative in this scenario. Which metric should guide the decision, and why is accuracy alone insufficient? Why do we also require a minimum recall?

We use the public [UCI Spambase dataset](https://archive.ics.uci.edu/dataset/94/spambase): 4,601 emails represented by 57 numerical measurements, including word/character frequencies and capital-letter run lengths. We do not need to process raw email text. This historical dataset is useful for learning, but does not represent today's complete email traffic.

In [ ]:
# SETUP — supplied. Run once.
from pathlib import Path
from io import BytesIO
from urllib.request import urlopen
from zipfile import ZipFile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.base import clone
from sklearn.naive_bayes import GaussianNB
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             confusion_matrix, ConfusionMatrixDisplay, log_loss)

DATA_URL = "https://archive.ics.uci.edu/static/public/94/spambase.zip"
# Optional offline fallback: upload the original UCI ZIP in Colab's Files panel.
local_zip = Path("spambase.zip")
if local_zip.exists():
    archive_bytes = local_zip.read_bytes()
else:
    try:
        with urlopen(DATA_URL, timeout=60) as response:
            archive_bytes = response.read()
    except Exception as exc:
        raise RuntimeError("Download failed. Retry, or upload spambase.zip from the UCI dataset page, then rerun.") from exc
with ZipFile(BytesIO(archive_bytes)) as archive:
    names_text = archive.read("spambase.names").decode("utf-8")
    feature_names = [line.split(":")[0] for line in names_text.splitlines()
                     if line.strip().endswith("continuous.") and not line.startswith("|")]
    data = pd.read_csv(archive.open("spambase.data"), header=None,
                       names=feature_names + ["spam"])
assert data.shape == (4601, 58), "Unexpected dataset version or column names."
assert not data.isna().any().any(), "Inspect missing values before continuing."
# Prevent identical labelled records from appearing on both sides of a split.
original_rows = len(data)
data = data.drop_duplicates().reset_index(drop=True)
X, y = data.drop(columns="spam"), data["spam"].astype(int)
assert set(y.unique()) == {0, 1}
print(f"Removed {original_rows - len(data)} exact duplicate rows; {len(data)} rows remain.")
display(data.head())



### YOUR TURN 1 — make one training/test split
Create a **stratified 80% training / 20% test** split using `train_test_split`, `test_size=0.20`, `stratify=y`, and `random_state=42`. The seed keeps team results comparable.

**The test set is now closed.** Do not inspect its class counts, plot it, or use it to select C or k. All model comparison and overfitting plots use only the training set. Within training, each CV fold temporarily holds out a different validation portion. There is no separate fixed validation set.

With **k=3, 4, and 5**, every training row is validated once per CV run. Use stratification so each fold roughly preserves the class proportions. Compare C values using the same folds within each k. We predeclare **k=5 as the main selection result**; k=3 and k=4 check whether that conclusion is sensitive to the fold count. Do not choose whichever k gives the most flattering score.

In [ ]:
# YOUR TURN 1 — complete the split before continuing.
X_train = X_test = y_train = y_test = None
# Replace the line above with:
# X_train, X_test, y_train, y_test = train_test_split(...)
assert X_train is not None, "Create the specified 80/20 stratified split."
assert len(X_train) == 3368 and len(X_test) == 842
print(f"Training: {len(X_train)} | Test (closed): {len(X_test)}")
display(y_train.value_counts().sort_index().rename_axis("class").to_frame("training count"))
print(f"Always predicting legitimate email gives {100*(y_train == 0).mean():.1f}% training accuracy,")
print("but catches no spam. Use ONLY X_train and y_train until the final evaluation.")

### Supplied evaluation and cross-validation helpers
Run this cell as supplied. The CV helper clones and fits a fresh model in each fold. For a pipeline, scaling is learned **inside that fold's training portion**. It returns per-fold scores, their means and sample standard deviations, and out-of-fold predictions (each row predicted by a model that did not train on it).

The confusion matrix has **actual classes as rows**, **predicted classes as columns**, and order `[0, 1]`: `[[TN, FP], [FN, TP]]`. Precision and recall refer to spam (class 1). Undefined precision is reported as 0 so an all-negative filter cannot win.

Fold standard deviation describes variation; it is not a confidence interval. Mean fold precision can differ from precision calculated from the pooled out-of-fold matrix. Use the **mean fold scores** for model selection.

In [ ]:
def evaluate(model, X_eval, y_eval, label, show_matrix=False):
    predictions = model.predict(X_eval)
    tn, fp, fn, tp = confusion_matrix(y_eval, predictions, labels=[0, 1]).ravel()
    result = {"model": label,
              "accuracy": accuracy_score(y_eval, predictions),
              "precision": precision_score(y_eval, predictions, zero_division=0),
              "recall": recall_score(y_eval, predictions, zero_division=0),
              "TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp)}
    if show_matrix:
        ConfusionMatrixDisplay.from_predictions(
            y_eval, predictions, labels=[0, 1],
            display_labels=["Legitimate (0)", "Spam (1)"], cmap="Blues")
        plt.title(label)
        plt.show()
    return result

K_values = [3, 4, 5]
def cross_validate_training(estimator, k):
    cv = StratifiedKFold(n_splits=k, shuffle=True, random_state=42)
    fold_rows = []
    oof_predictions = np.empty(len(y_train), dtype=int)
    for fold, (fit_idx, valid_idx) in enumerate(cv.split(X_train, y_train), start=1):
        model = clone(estimator)
        X_fit, X_valid = X_train.iloc[fit_idx], X_train.iloc[valid_idx]
        y_fit, y_valid = y_train.iloc[fit_idx], y_train.iloc[valid_idx]
        model.fit(X_fit, y_fit)
        row = {"k": k, "fold": fold}
        for partition, X_part, y_part in [("train", X_fit, y_fit),
                                          ("valid", X_valid, y_valid)]:
            predictions = model.predict(X_part)
            probabilities = model.predict_proba(X_part)
            row[f"{partition}_precision"] = precision_score(y_part, predictions, zero_division=0)
            row[f"{partition}_recall"] = recall_score(y_part, predictions, zero_division=0)
            row[f"{partition}_accuracy"] = accuracy_score(y_part, predictions)
            row[f"{partition}_log_loss"] = log_loss(y_part, probabilities, labels=[0, 1])
            if partition == "valid":
                oof_predictions[valid_idx] = predictions
        fold_rows.append(row)
    folds = pd.DataFrame(fold_rows)
    summary = {"k": k}
    for metric in ["train_precision", "valid_precision", "train_recall", "valid_recall",
                   "train_accuracy", "valid_accuracy", "train_log_loss", "valid_log_loss"]:
        summary[metric + "_mean"] = folds[metric].mean()
        summary[metric + "_std"] = folds[metric].std(ddof=1)
    return summary, folds, oof_predictions


## 2. Cross-validate the Naive Bayes baseline · 8–20 minutes

Use **`GaussianNB()`** for the numerical measurements. It assumes conditional independence of the features given the class, and a Gaussian feature distribution within each class. These assumptions can be imperfect for word frequencies.

**YOUR TURN 2:** create the estimator. The helper fits fresh copies using **k=3, 4, and 5 on the training set only**. Do not fit once on all training rows before cross-validation. The supplied code shows the five-fold out-of-fold confusion matrix, not a test matrix.

**Q2 — Baseline evidence.** Report mean CV precision and recall for all three k values. Read FP and FN from the five-fold out-of-fold matrix. Does this baseline suit our goal? Does your judgment change with k? Explain using the errors, not just accuracy.

In [ ]:
# YOUR TURN 2 — create the Naive Bayes estimator.
nb_model = None  # Replace with GaussianNB()
assert nb_model is not None, "Create your GaussianNB baseline first."
nb_rows, nb_fold_tables, nb_oof = [], [], {}
for k in K_values:
    summary, folds, predictions = cross_validate_training(nb_model, k)
    nb_rows.append(summary)
    nb_fold_tables.append(folds)
    nb_oof[k] = predictions
nb_comparison = pd.DataFrame(nb_rows)
display(nb_comparison[["k", "valid_precision_mean", "valid_precision_std",
                       "valid_recall_mean", "valid_recall_std", "valid_accuracy_mean"]].round(4))
ConfusionMatrixDisplay.from_predictions(
    y_train, nb_oof[5], labels=[0, 1], display_labels=["Legitimate (0)", "Spam (1)"], cmap="Blues")
plt.title("Naive Bayes — 5-fold out-of-fold predictions on TRAINING data")
plt.show()

## 3. Regularization and overfitting curves · 20–45 minutes

**Smaller C means stronger L2 regularization**; larger C permits larger coefficients. `StandardScaler` stays inside the pipeline so every fold learns its scaling from its own fitting rows.

**Predict first:** as regularization becomes weaker, will fitting performance and held-out fold performance improve together?

**YOUR TURN 3:** enter **`[0.001, 0.01, 0.1, 1, 10, 100, 1000]`**. The supplied loop runs all seven settings with **k=3, 4, and 5**. Each k uses the same folds for every C and for the baseline. The default probability threshold remains 0.5. This is a small, fast experiment; no additional model search is required.

Run the plot cell after the table. It draws **training versus CV validation curves across C**, separately for each k:
- **Precision (higher is better)** connects the experiment to our filtering goal.
- **Log loss (lower is better)** measures the quality of probability predictions and penalizes confidently wrong predictions. It can reveal overfitting even when the predicted classes change very little.

Look for a widening train–validation gap as C grows, particularly when training loss improves but validation loss worsens. Do not claim overfitting just because training scores are better; use the trend and the fold variation. These are **validation curves over regularization strength**, not learning curves over sample size or optimizer iteration plots.

**Q3 — Cross-validation and overfitting.** Include the CV comparison table and the curve figure. Explain (a) which C is most strongly regularized, (b) what the training/validation curves show about underfitting or overfitting, using specific C values, (c) whether the conclusion changes for k=3, 4, and 5, including fold variation, and (d) why scaling must happen inside each fold. A trend may be weak: describe what your evidence actually supports.

In [ ]:
# YOUR TURN 3 — enter the regularization values.
C_values = []  # Replace with [0.001, 0.01, 0.1, 1, 10, 100, 1000]
assert C_values == [0.001, 0.01, 0.1, 1, 10, 100, 1000]

def make_logistic(C):
    # liblinear uses L2 regularization by default.
    return make_pipeline(StandardScaler(),
                         LogisticRegression(C=C, solver="liblinear", max_iter=2000,
                                            random_state=42))

rows, fold_tables = [], []
for k in K_values:
    for C in C_values:
        summary, folds, _ = cross_validate_training(make_logistic(C), k)
        summary["C"] = C
        folds["C"] = C
        rows.append(summary)
        fold_tables.append(folds)
comparison = pd.DataFrame(rows)
fold_results = pd.concat(fold_tables, ignore_index=True)
display(comparison[["k", "C", "train_precision_mean", "valid_precision_mean",
                    "valid_precision_std", "valid_recall_mean", "valid_recall_std",
                    "train_log_loss_mean", "valid_log_loss_mean"]].round(4))
# Inspect an individual fold result when you need to explain unexpected variation.
# display(fold_results)


In [ ]:
# SUPPLIED PLOTS — training versus held-out CV folds, using no test data.
fig, axes = plt.subplots(2, 3, figsize=(15, 8), sharex=True)
for column, k in enumerate(K_values):
    results = comparison[comparison["k"] == k].sort_values("C")
    for row, (metric, label) in enumerate([("precision", "Precision (higher is better)"),
                                           ("log_loss", "Log loss (lower is better)")]):
        ax = axes[row, column]
        ax.plot(results["C"], results[f"train_{metric}_mean"], "o--", label="Training-fold mean")
        ax.errorbar(results["C"], results[f"valid_{metric}_mean"],
                    yerr=results[f"valid_{metric}_std"], fmt="o-", capsize=3,
                    label="CV validation mean ± fold SD")
        ax.set_xscale("log")
        ax.set_title(f"{k}-fold CV")
        ax.set_ylabel(label)
        ax.set_xlabel("C: stronger regularization ← → weaker regularization")
        ax.grid(alpha=0.3)
        ax.legend(fontsize=8)
fig.suptitle("Regularization and overfitting — TRAINING set only", fontsize=15)
plt.tight_layout()
plt.show()


## 4. Choose on training CV, then open the unseen test set · 45–55 minutes

Use the predeclared **five-fold** results to choose C: highest mean CV precision among settings with mean CV recall ≥ 0.80. If precision ties, prefer higher recall, then smaller C. Use unrounded values. If no setting qualifies, report failure and choose the highest-recall setting for diagnosis; do not claim it meets the requirement.

Use **k=3 and k=4 as sensitivity checks**, not as extra independent datasets or a contest to find the best-looking k. The fold training sizes and partitions differ. If the preferred C changes or differences are small relative to fold variation, acknowledge uncertainty. Log-loss curves diagnose overfitting; precision with the recall safeguard remains our stated selection rule.

**Q4 — Decision before testing.** Record your chosen C and five-fold mean precision/recall. Compare it with the Naive Bayes baseline using the same five-fold CV. State whether the three- and four-fold results and overfitting curves support a stable choice. Explain the precision/recall tradeoff. Write this answer **before** running the final evaluation.

**YOUR TURN 4:** record your chosen C and reason. The final cell fits that pipeline on **all of X_train**, then evaluates it **once** on the untouched test set. Do not use the test result to choose C, k, features, or thresholds.

In [ ]:
# YOUR TURN 4 — record your decision BEFORE the final evaluation.
chosen_C = None  # Replace with a C value from your comparison.
decision_reason = ""  # Summarize the training CV evidence supporting your choice.
assert chosen_C in C_values, "Choose a C value using training CV evidence."
assert decision_reason.strip(), "Record your reason before opening the test set."
print("Chosen C:", chosen_C)
print("Reason:", decision_reason)

In [ ]:
# FINAL EVALUATION — supplied. Run only after recording Q4.
final_model = make_logistic(chosen_C)
final_model.fit(X_train, y_train)
test_result = evaluate(final_model, X_test, y_test,
                       f"Chosen logistic model (C={chosen_C}) — TEST", True)
display(pd.DataFrame([test_result]).round(4))

**Q5 — Final evaluation.** Include the test confusion matrix and test precision/recall. Explain FP and FN using their actual counts. Compare the test results with the five-fold CV estimates and state whether the evidence supports your recommendation. Why must you avoid choosing another C or k using these test results? A test score is an estimate, not a guarantee.

## 5. Finish your Word answers and submit · 55–75 minutes

**Q6 — Reflection and limits.** Describe one useful contribution from your Thinking Partner and how you checked it (or explain your own debugging/check if you did not use AI). Name one limitation of this experiment, and one next investigation. No extra experiment is required.

### Your Word document
Use the title **Challenge 2: Can We Trust the Spam Filter?**, your **team name**, **members and student IDs**, then headings **Q1–Q6**. Write concise group answers with the requested evidence; use short paragraphs, allowing extra space for tables and figures. Copy the CV comparison table, overfitting curves and relevant confusion matrices from your notebook. Do not paste an AI transcript.

### Submission checklist
- [ ] Six numbered answers are in **`Challenge-2_TeamName_Answers.docx`**.
- [ ] The notebook contains your completed code, tables, matrices and recorded decision. In Colab use **File → Download → Download .ipynb** to save **`Challenge-2_TeamName.ipynb`** with outputs.
- [ ] Put those **two files** in **`Challenge-2_TeamName.zip`**. Use your assigned Moodle team name (replace spaces with hyphens).
- [ ] Open the ZIP to verify both files are present; one team member uploads it to **Challenge 2 Submission** on Moodle before its displayed deadline.

**If time runs short:** submit your current notebook and answer what you can with evidence. State unfinished work honestly. Clear reasoning and a working core experiment earn credit; you do not need a perfect model. Complete the required k=3, 4, 5 comparison and overfitting plots before considering any extra experiments.

**How the work is evaluated (20 points):** problem/metric reasoning (4), baseline and confusion matrix (4), cross-validation and overfitting experiment (4), model choice and honest test interpretation (4), clear reproducible submission and reflection (4).

### Sources
- Hopkins, M., Reeber, E., Forman, G., & Suermondt, J. (1999). [Spambase](https://doi.org/10.24432/C53G6X). UCI Machine Learning Repository, [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/). Exact duplicate labelled rows are removed for this experiment.
- scikit-learn: [GaussianNB](https://scikit-learn.org/stable/modules/generated/sklearn.naive_bayes.GaussianNB.html), [LogisticRegression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html), [Pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html).